# Actividad 16 Tipos

Vamos a ver como usar python para obtener datos de una API y almacenarlos en estrucsturas adecuadas

Ejecuta los siguientes ejemplos y resnponde a las cuestiones

# 1. Ejecuta y revisa los siguientes ejemplos


In [1]:
import json
import pandas as pd
import requests

# monedas y divisa destino
COINS = "bitcoin,ethereum,solana"
VS = "usd"

url = "https://api.coingecko.com/api/v3/simple/price"
params = {"ids": COINS, "vs_currencies": VS}

resp = requests.get(url, params=params, timeout=10)
data = resp.json()

print(json.dumps(data, indent=2, ensure_ascii=False))

print("=== Precios actuales (USD) ===")
print("Bitcoin:", data["bitcoin"]["usd"], "USD")
print("Ethereum:", data["ethereum"]["usd"], "USD")
print("Solana:", data["solana"]["usd"], "USD")

{
  "bitcoin": {
    "usd": 80604
  },
  "ethereum": {
    "usd": 2354.02
  },
  "solana": {
    "usd": 101.85
  }
}
=== Precios actuales (USD) ===
Bitcoin: 80604 USD
Ethereum: 2354.02 USD
Solana: 101.85 USD


Ahora incorporamos un FOR y WHILE para visualizar

In [ ]:
import requests

# Monedas y divisa
COINS = "bitcoin,ethereum,solana"
VS = "usd"

url = "https://api.coingecko.com/api/v3/simple/price"
params = {"ids": COINS, "vs_currencies": VS}

resp = requests.get(url, params=params, timeout=10)
data = resp.json()

print("=== Usando for ===")
for coin, valores in data.items():
    print(f"{coin.capitalize()}: {valores[VS]} {VS.upper()}")

print("\n=== Usando while ===")
monedas = list(data.items())   # convertimos en lista de pares (moneda, valores)
i = 0
while i < len(monedas):
    coin, valores = monedas[i]
    print(f"{coin.capitalize()}: {valores[VS]} {VS.upper()}")
    i += 1

Ahora mostramos la estructura del resultado

In [ ]:
import requests

# Monedas y divisa
COINS = "bitcoin,ethereum,solana"
VS = "usd"

url = "https://api.coingecko.com/api/v3/simple/price"
params = {"ids": COINS, "vs_currencies": VS}

resp = requests.get(url, params=params, timeout=10)
data = resp.json()

print("=== JSON completo ===")
print(data)

print("\n=== Claves principales ===")
print(list(data.keys()))

print("\n=== Claves internas (por cada moneda) ===")
for coin, valores in data.items():
    print(f"{coin}: {list(valores.keys())}")

=== JSON completo ===
{'bitcoin': {'usd': 82133}, 'ethereum': {'usd': 2404.0}, 'solana': {'usd': 104.45}}

=== Claves principales ===
['bitcoin', 'ethereum', 'solana']

=== Claves internas (por cada moneda) ===
bitcoin: ['usd']
ethereum: ['usd']
solana: ['usd']


Ahora un ejemplo más detallado

In [ ]:
import requests

url = "https://api.binance.com/api/v3/ticker/price"

resp = requests.get(url, timeout=10)
if resp.status_code != 200:
    print("Error:", resp.status_code, resp.text)
    exit()

data = resp.json()

print("=== Número total de pares devueltos ===")
print(len(data))

print("\n=== Primeros 5 registros (JSON bruto) ===")
for item in data[:5]:
    print(item)

print("\n=== Claves de cada registro ===")
if data:
    print(list(data[0].keys()))  # deberían ser ['symbol', 'price']

print("\n=== Todos los símbolos (solo primeros 10) ===")
for item in data[:10]:
    print(item["symbol"])

## 2. Haz lo necesario para obtener un dataframe con los valores históricos de BTC en USD

In [ ]:
import json
import pandas as pd
import requests


url = "https://api.binance.com/api/v3/klines"
params = {"symbol": "BTCUSDT", "interval": "1d", "limit": 100}

response = requests.get(url, params=params, timeout=10)
if response.status_code != 200:
    print("Error en la petición:", response.status_code, response.text)
    exit()

data_json = json.loads(response.text)

columns = [
    "Open_Time",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Close_Time",
    "Quote_Asset_Volume",
    "Number_of_Trades",
    "Taker_Buy_Base_Asset_Volume",
    "Taker_Buy_Quote_Asset_Volume",
    "Ignore"
]

df = pd.DataFrame(data_json, columns=columns)
print(df.head())

df["Open_Time"] = pd.to_datetime(df["Open_Time"], unit="ms")

df["Close_Time"] = pd.to_datetime(df["Close_Time"], unit="ms")
numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

df_btc = df[["Open_Time", "Open", "High", "Low", "Close", "Volume"]]

print("=== DataFrame de Precios Históricos de BTC/USD ===")
print(df_btc.head(10))

In [10]:
import requests
import pandas as pd
import json

COIN = "bitcoin"
VS = "usd"
TIME="max"
INTERVALO ="daily"

#url = ("https://api.coingecko.com/api/v3/coins/bitcoin/market_chart")
#params = {"vs_currency":VS, "days":TIME,"interval":INTERVALO}

#resp = requests.get(url, params=params, timeout=10)
#if resp.status_code != 200:
#    print("Error: ", resp.status_code, resp.text)
#    exit()
#datos = resp.json()

with open("btc_klines.json", "r", encoding="utf-8") as f:
    datos = json.load(f)

columns = [
    "Open_Time","Open","High","Low","Close","Volume","Close_Time",
    "Quote_Asset_Volume","Number_of_Trades","Taker_Buy_Base_Asset_Volume",
    "Taker_Buy_Quote_Asset_Volume","Ignore"
]

dfBTC = pd.DataFrame(datos, columns=columns)
#print(dfBTC.head())

df_btc = dfBTC[["Open", "High", "Low", "Close", "Volume"]]
print(df_btc)

print("=== JSON completo ===")
print(datos)

print("\n=== Claves principales ===")
print(list(datos))



          Open       High        Low      Close       Volume
0     68000.00   68347.77   67500.44   67653.40  22932.93211
1     67653.40   68940.82   66475.90   67208.47  14582.15270
2     67208.47   67312.04   66120.75   66711.18  30498.15524
3     66711.18   68648.41   66405.92   67743.95  24157.91952
4     67743.95   68061.12   66122.78   66325.37  31320.17994
..         ...        ...        ...        ...          ...
360  245579.57  251176.50  244726.01  250790.64  14320.73824
361  250790.64  254024.32  249101.82  249378.98  15801.51986
362  249378.98  252086.59  235747.55  236219.84  34050.17326
363  236219.84  237413.64  223968.17  225675.15  34859.97288
364  225675.15  227916.97  221404.19  227547.56  20915.30423

[365 rows x 5 columns]
=== JSON completo ===
[[1760918400000, '68000.00', '68347.77', '67500.44', '67653.40', '22932.93211', 1761004799999, '1555465146.72', 2433810, '11466.46606', '777732573.36', '0'], [1761004800000, '67653.40', '68940.82', '66475.90', '67208.47', 

## 3. Haz que los índices del df anterior sean numéricos

## 4. Haz una copia del df en la que los índices sean las fechas
